# 03. representation collapse와 평가 설계

목표: 사전학습 loss만 보지 않고 embedding 다양성을 진단하며, 모달리티별 실험 카드를 만든다. 외부 패키지는 필요 없다.

In [ ]:
import math
import statistics

def cosine_similarity(left, right):
    dot = sum(a * b for a, b in zip(left, right))
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    return dot / (left_norm * right_norm + 1e-12)

def representation_report(embeddings, variance_threshold=1e-4):
    # 차원별 분산이 모두 작고 샘플 간 cosine이 1에 가까우면 collapse를 의심한다.
    dimensions = list(zip(*embeddings))
    variances = [statistics.pvariance(values) for values in dimensions]
    pairwise = [
        cosine_similarity(embeddings[i], embeddings[j])
        for i in range(len(embeddings))
        for j in range(i + 1, len(embeddings))
    ]
    return {
        "mean_dimension_variance": sum(variances) / len(variances),
        "active_dimensions": sum(v > variance_threshold for v in variances),
        "mean_pairwise_cosine": sum(pairwise) / len(pairwise),
    }

In [ ]:
diverse = [
    [1.0, 0.0, 0.2],
    [0.0, 1.0, -0.1],
    [-0.8, 0.1, 1.0],
    [0.3, -0.9, 0.4],
]
collapsed = [
    [1.0000, 1.0000, 1.0000],
    [1.0001, 0.9999, 1.0000],
    [0.9999, 1.0001, 1.0000],
    [1.0000, 1.0000, 1.0001],
]
print("다양한 표현:", representation_report(diverse))
print("붕괴 후보:", representation_report(collapsed))

분산과 cosine은 경보 지표이지 collapse의 완전한 증명이 아니다. batch·dataset 전체의 통계, covariance spectrum, effective rank, k-NN/linear probe를 함께 봐야 한다. 특히 정규화된 표현은 분산의 절대 척도가 달라질 수 있다.

In [ ]:
def experiment_card(modality, token, context, target, downstream_metric):
    """새 JEPA 실험에서 빠뜨리기 쉬운 결정을 한 장으로 만든다."""
    return {
        "modality": modality,
        "token_or_patch": token,
        "context_rule": context,
        "target_rule": target,
        "downstream_metric": downstream_metric,
        "required_diagnostics": [
            "pretraining loss",
            "dimension variance",
            "pairwise similarity",
            "linear probe",
            "external-domain transfer",
        ],
    }

card = experiment_card(
    modality="server metrics table",
    token="metric column × time window",
    context="과거 정상 구간의 일부 metric",
    target="숨긴 metric subset의 미래 latent",
    downstream_metric="anomaly AUROC + detection delay",
)
for key, value in card.items():
    print(f"{key}: {value}")

## 심화 과제

1. 자신의 데이터용 experiment card를 만든다.
2. reconstruction baseline과 동일한 encoder·compute budget으로 비교한다.
3. random mask와 구조 인식 mask를 ablation한다.
4. 사전학습과 다른 cohort·domain에서 frozen linear probe를 수행한다.
5. 성능뿐 아니라 데이터 누설, 민감 속성, 계산량과 실패 사례를 기록한다.